In [1]:
import json
from pathlib import Path


In [ ]:
if "g2p_clean" not in globals():
    %run annotation.ipynb
    %run classifier.ipynb


In [ ]:
# --------------------------------------------------
# Patient diagnosis (asked first, so the (gene, disease) pair can be 
# validated against Cardiac_G2P as soon as the variant resolves a gene)
# --------------------------------------------------
'''
# A dropdown, not a numbered menu -- so the disease name is directly visible
# and doesn't need looking up against an index number. Placeholder first
# option ("-- select a diagnosis --", value=None) means nothing is recorded
# until the user actively picks something -- ipywidgets Dropdown otherwise
# defaults its .value to the first real option.
# Batch mode: if _VALIDATION_ANSWERS is set in the namespace (by
# validation.ipynb, via `%run -i main1.ipynb`), skip the widget entirely and
# take the diagnosis straight from it -- lets validation.ipynb drive this
# exact notebook non-interactively instead of re-implementing its logic.
_batch = globals().get("_VALIDATION_ANSWERS")
'''

if _batch is not None:
    disease_short_code = _batch.get("disease_short_code")
    disease_long_name = (
        disease_ref.loc[disease_ref["dis_name"] == disease_short_code, "dis_long_name"].iloc[0]
        if disease_short_code else "Unknown / other"
    )
    print(f"[batch] Recorded diagnosis: {disease_long_name}" + (f" ({disease_short_code})" if disease_short_code else ""))
else:
    import ipywidgets as widgets
    from IPython.display import display

    _disease_options = (
        [("-- select a diagnosis --", None)]
        + [(f"{row.dis_long_name} ({row.dis_name})", row.dis_name) for row in disease_ref.itertuples()]
        + [("Unknown / other", None)]
    )

    disease_short_code = None
    disease_long_name  = "Unknown / other"

    _disease_dropdown = widgets.Dropdown(
        options=_disease_options,
        value=None,
        description="Diagnosis:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="700px"),
    )
    _disease_output = widgets.Output()

    def _on_disease_change(change):
        global disease_short_code, disease_long_name
        if change["name"] != "value":
            return
        disease_short_code = change["new"]
        disease_long_name = (
            disease_ref.loc[disease_ref["dis_name"] == disease_short_code, "dis_long_name"].iloc[0]
            if disease_short_code else "Unknown / other"
        )
        with _disease_output:
            _disease_output.clear_output()
            print(f"Recorded diagnosis: {disease_long_name}" + (f" ({disease_short_code})" if disease_short_code else ""))

    _disease_dropdown.observe(_on_disease_change, names="value")

    print("Patient's diagnosis / referral indication:")
    print("Select from the dropdown below, then wait for 'Recorded diagnosis: ...' before running the next cell.")
    display(_disease_dropdown, _disease_output)


NameError: name '_batch' is not defined

In [ ]:
_batch = globals().get("_VALIDATION_ANSWERS")

variant = _batch["variant"] if _batch is not None else input("Enter HGVS variant: ").strip()
if not variant:
    raise ValueError("No variant entered.")

basename = (
    variant.replace(":", "_").replace("(", "_")
           .replace(")", "_").replace("/", "_").replace(" ", "_")
)

output_json, table_rows = annotate_variant(variant)
if output_json is None:
    raise ValueError("Annotation failed — no output returned from VEP.")

json_path = Path(f"outputs/{basename}.json")
tsv_path  = Path(f"outputs/{basename}.tsv")
write_json_file(output_json, json_path)
write_tsv_file(table_rows, tsv_path)
print("Saved JSON:", json_path)
print("Saved TSV: ", tsv_path)


In [5]:
context = build_classification_context(output_json, g2p_clean, disease_short_code)

print(f"Gene:   {context['gene_symbol']}")
print(f"ICC gene (Cardiac G2P): {context['icc_gene']}")
if context['icc_gene']:
    # Show every G2P entry for this gene (all diseases it's curated for),
    # not just the confirmed-disease subset the ACMG rules actually use --
    # context["g2p_hits"] is disease-scoped now, so an unconfirmed pair
    # would otherwise print an empty table with no explanation.
    cols = [c for c in ["gene_symbol", "disease_name", "gene_disease_validity", "expert_panel"]
            if c in context["all_g2p_hits"].columns]
    print(context["all_g2p_hits"][cols].to_string(index=False))

print()
gdp = context["gene_disease_pair"]
if gdp["pair_found"]:
    print(f"Gene-disease pair CONFIRMED: {context['gene_symbol']} -- {gdp['g2p_referral_indication']}")
else:
    print(f"Gene-disease pair NOT CONFIRMED: {gdp['reason']}")


Gene:   MYBPC3
ICC gene (Cardiac G2P): True
GENE SYMBOL               DISEASE LABEL CLASSIFICATION                                                         GCEP
     MYBPC3 hypertrophic cardiomyopathy     Definitive Hereditary Cardiovascular Disease Gene Curation Expert Panel


In [ ]:
# --------------------------------------------------
# Manual clinical evidence (not computable from any API/reference file)
# --------------------------------------------------
_batch = globals().get("_VALIDATION_ANSWERS")

print("De novo status (PS2/PM6) -- based on family testing, not computable from public data.")
de_novo_answer = (
    _batch.get("de_novo_answer", "unknown") if _batch is not None
    else input("Is this variant de novo (confirmed absent in both biological parents)? [y/n/unknown]: ").strip().lower()
)

manual_evidence = {
    "de_novo_status": "unknown",
    "zygosity": "het",
    "disease_context": disease_short_code.lower() if disease_short_code else None,
}

if de_novo_answer == "y":
    parentage_answer = (
        _batch.get("parentage_answer", "n") if _batch is not None
        else input("Was biological parentage (maternity AND paternity) confirmed by testing? [y/n]: ").strip().lower()
    )
    manual_evidence["de_novo_status"] = "confirmed" if parentage_answer == "y" else "assumed"
elif de_novo_answer == "n":
    manual_evidence["de_novo_status"] = "not_de_novo"

print()
print("Zygosity (BA1/BS1 -- only affects genes with a recessive/biallelic mechanism).")
zygosity_answer = (
    _batch.get("zygosity_answer", "het") if _batch is not None
    else input("Is this variant heterozygous or homozygous in the patient? [het/hom]: ").strip().lower()
)
if zygosity_answer in ("hom", "homozygous"):
    manual_evidence["zygosity"] = "hom"

context["manual_evidence"] = manual_evidence
print(f"Recorded de novo status: {manual_evidence['de_novo_status']}")
print(f"Recorded zygosity:       {manual_evidence['zygosity']}")
print(f"Recorded disease context: {manual_evidence['disease_context']}")


In [ ]:
'''

Curator-only evidence (PM3, PP1, PP4, BS2, BS4, BP2, BP5) -- patient-
specific clinical facts a curated literature table can't supply (phasing,
segregation, phenotype specificity), collected the same way as de novo
status/zygosity above.
Only 4 questions cover all 7 codes: phasing and segregation each route to
one of two codes using this gene's inheritance/penetrance facts already in
Cardiac_G2P (not asked twice) -- see apply_curator_only_rules/
_manual_curator_hits in classifier.ipynb for the exact routing logic.

'''

_batch = globals().get("_VALIDATION_ANSWERS")

print("Curator-only evidence -- patient-specific facts, not in any lookup table.")
print()

print("Phasing (PM3/BP2):")
phasing_answer = (
    _batch.get("phasing_answer", "unknown") if _batch is not None
    else input("Is there a second (likely) pathogenic variant identified in this same gene "
               "in this patient? [y/n/unknown]: ").strip().lower()
)

phasing_relationship = None
if phasing_answer == "y":
    phasing_relationship = (
        _batch.get("phasing_relationship") if _batch is not None
        else input("Is this variant in cis or trans with that second variant? [cis/trans/unknown]: ").strip().lower()
    )

print()
print("Segregation (PP1/BS4):")
segregation_answer = (
    _batch.get("segregation_answer", "unknown") if _batch is not None
    else input("Has segregation with disease been assessed across multiple family members? [y/n/unknown]: ").strip().lower()
)

segregation_result = None
segregation_meioses = None
if segregation_answer == "y":
    if _batch is not None:
        segregation_result = _batch.get("segregation_result")
        segregation_meioses = _batch.get("segregation_meioses")
    else:
        seg_detail = input(
            "Does the variant segregate with disease (present in affected, absent in unaffected)? [y/n/unknown]: "
        ).strip().lower()
        # Perl never defaulted an ambiguous/absent answer to firing a code --
        # PP1/BS4 were curator-only, initialised to 0, only ever set by an
        # explicit, affirmative curator action (see ACMG_add_evidence_ext.php).
        # A non-y/n answer here must fall through to "not assessed" (no hit),
        # matching every other question in this cell -- not silently default
        # to "does_not_segregate", which would incorrectly fire BS4.
        if seg_detail == "y":
            segregation_result = "segregates"
            meioses_raw = input(
                "How many informative meioses/segregations support this? "
                "(ClinGen Cardiomyopathy VCEP: >=7 Strong, >=5 Moderate, >=3 Supporting, "
                "<3 does not meet PP1 -- leave blank if unknown): "
            ).strip()
            segregation_meioses = int(meioses_raw) if meioses_raw.isdigit() else None
        elif seg_detail == "n":
            segregation_result = "does_not_segregate"
            meioses_raw = input(
                "How many non-segregating individuals (genotype+/phenotype- or genotype-/"
                "phenotype+) were observed? (ClinGen requires >=2 to apply BS4 alone -- "
                "leave blank if unknown): "
            ).strip()
            segregation_meioses = int(meioses_raw) if meioses_raw.isdigit() else None
        else:
            segregation_result = None

print()
pp4_answer = (
    _batch.get("pp4_answer", "unknown") if _batch is not None
    else input("PP4: is the patient's phenotype/family history highly specific for this gene's "
               "disease (no equally plausible alternate diagnosis)? [y/n/unknown]: ").strip().lower()
)

print()
_penetrance_note = _g2p_penetrance_note(context.get("g2p_hits"))
if _penetrance_note:
    print(f"BS2: skipped -- Cardiac_G2P flags this gene/disease as '{_penetrance_note}', "
          f"so 'full penetrance expected' (required for BS2) doesn't hold.")
    bs2_answer = "unknown"
else:
    bs2_answer = (
        _batch.get("bs2_answer", "unknown") if _batch is not None
        else input("BS2: has this variant been observed in a phenotype-negative adult, past the "
                   "expected age of onset, for a gene/inheritance pattern with full penetrance "
                   "expected? [y/n/unknown]: ").strip().lower()
    )

print()
bp5_answer = (
    _batch.get("bp5_answer", "unknown") if _batch is not None
    else input("BP5: was a variant identified in a DIFFERENT gene that already fully explains "
               "this patient's phenotype? [y/n/unknown]: ").strip().lower()
)

context["manual_evidence"].update({
    "phasing_second_variant":    "yes" if phasing_answer == "y" else ("no" if phasing_answer == "n" else "unknown"),
    "phasing_relationship":      phasing_relationship,
    "segregation_assessed":      "yes" if segregation_answer == "y" else ("no" if segregation_answer == "n" else "unknown"),
    "segregation_result":        segregation_result,
    "segregation_meioses":       segregation_meioses,
    "phenotype_specific":        "yes" if pp4_answer == "y" else ("no" if pp4_answer == "n" else "unknown"),
    "healthy_adult_observed":    "yes" if bs2_answer == "y" else ("no" if bs2_answer == "n" else "unknown"),
    "alternate_molecular_basis": "yes" if bp5_answer == "y" else ("no" if bp5_answer == "n" else "unknown"),
})

print()
print("Recorded curator-only evidence:")
for _k in ("phasing_second_variant", "phasing_relationship", "segregation_assessed",
           "segregation_result", "segregation_meioses", "phenotype_specific",
           "healthy_adult_observed", "alternate_molecular_basis"):
    print(f"  {_k}: {context['manual_evidence'][_k]}")


## ACMG/AMP Evidence (ClinGen CSpec-aware)
Thresholds from ClinGen CSpec registry where available; generic ACMG/AMP defaults otherwise.


In [7]:
freq_rules = apply_population_rules(context)
print(f"Threshold source:  {freq_rules['cspec_source']}")
print(f"AF used:           {freq_rules['af_used']}  (from {freq_rules['af_source']})")
print(f"Zygosity:          {freq_rules['zygosity']}  (biallelic/recessive gene: {freq_rules['is_biallelic_gene']})")
if freq_rules['zygosity_adjusted']:
    print("  -> BA1/BS1 thresholds sqrt-adjusted for homozygous recessive variant")
print(f"BA1 threshold: >= {freq_rules['ba1_threshold']}")
print(f"BS1 threshold: >= {freq_rules['bs1_threshold']}")
print(f"PM2 threshold: <= {freq_rules['pm2_max']}  (strength: {freq_rules['pm2_strength']})")
print()
print(f"BA1 (stand-alone benign):    {freq_rules['BA1']}")
print(f"BS1 (strong benign):         {freq_rules['BS1']}")
print(f"PM2 ({freq_rules['pm2_strength']} pathogenic):" + " " * max(0, 12 - len(freq_rules['pm2_strength'])) + f" {freq_rules['PM2']}")


Threshold source:  ClinGen CSpec GN095
AF used:           1.313e-05  (from gnomad_global/gnomadg)
Zygosity:          het  (biallelic/recessive gene: False)
BA1 threshold: >= 0.001
BS1 threshold: >= 0.0002
PM2 threshold: <= 4e-05

BA1 (stand-alone benign):    False
BS1 (strong benign):         False
PM2 (supporting pathogenic): True


In [8]:
pvs1_rules = apply_pvs1_rule(context)
print(f"Consequence:   {', '.join(pvs1_rules['consequence_terms'])}")
print(f"Null variant:  {pvs1_rules['is_null_variant']}")
print(f"LOF in G2P:    {pvs1_rules['lof_mechanism']}")
print()
print(f"PVS1 (very strong pathogenic): {pvs1_rules['PVS1']}")
if pvs1_rules['is_null_variant'] and not pvs1_rules['lof_mechanism']:
    print("  Note: null variant but LOF not the G2P mechanism (e.g. dominant negative)")
if pvs1_rules.get('ttn_gate'):
    tg = pvs1_rules['ttn_gate']
    print(f"  TTN exon gate: exon {tg['exon_te_id']} ({tg['exon_domain'] or 'no domain annotation'}), "
          f"PSI(DCM)={tg['psi_dcm']}, terminal={tg['is_terminal_exon']}  => gate_pass={tg['gate_pass']}")
    if tg['reason']:
        print(f"    {tg['reason']}")


Consequence:   stop_gained
Null variant:  True
LOF in G2P:    True

PVS1 (very strong pathogenic): True


In [9]:
comp_rules = apply_computational_rules(context)
revel = comp_rules['revel_score']
print(f"Threshold source:  {comp_rules['cspec_source']}")
print(f"Consequence eligible for REVEL: {comp_rules['variant_type_eligible']}")
print(f"REVEL score:   {revel if revel is not None else 'not available'}")
print(f"PP3 threshold: >= {comp_rules['pp3_threshold']}")
print(f"BP4 threshold: <= {comp_rules['bp4_threshold']}")
print()
print(f"PP3 (supporting pathogenic): {comp_rules['PP3']}")
print(f"BP4 (supporting benign):     {comp_rules['BP4']}")


Threshold source:  ClinGen CSpec GN095
Consequence eligible for REVEL: False
REVEL score:   not available
PP3 threshold: >= 0.7
BP4 threshold: <= 0.4

PP3 (supporting pathogenic): False
BP4 (supporting benign):     False


In [10]:
pm1_rules = apply_pm1_rule(context)
print(f"Matched region: {pm1_rules['matched_region'] or '(none)'}")
if pm1_rules['pmid']:
    print(f"Reference:      {pm1_rules['pmid']}")
print(f"Assessed:       {pm1_rules['have_assessed']}")
if pm1_rules['reason']:
    print(f"Reason:         {pm1_rules['reason']}")
print()
pm1_label = f"  [{pm1_rules['PM1_strength']}]" if pm1_rules['PM1'] else ""
pp2_label = f"  [{pm1_rules['PP2_strength']}]" if pm1_rules['PP2'] else ""
print(f"PM1 (hotspot/domain):        {pm1_rules['PM1']}{pm1_label}")
print(f"PP2 (whole-gene, no region): {pm1_rules['PP2']}{pp2_label}")


Matched region: (none)
Assessed:       False
Reason:         not a missense variant

PM1 (hotspot/domain):        False
PP2 (whole-gene, no region): False


In [11]:
ps1_rules = apply_ps1_rule(context)
bp1_rules = apply_bp1_rule(context)

print("PS1 (same amino acid change, previously P/LP in ClinVar):")
print(f"  Position:       {ps1_rules['position']}")
print(f"  Current change: {ps1_rules['current_change']}")
if ps1_rules['ps1_hits']:
    print(f"  Matches:        {ps1_rules['ps1_hits']}")
if ps1_rules['reason']:
    print(f"  Reason:         {ps1_rules['reason']}")
ps1_label = f"  [{ps1_rules['PS1_strength']}]" if ps1_rules['PS1'] else ""
print(f"  PS1: {ps1_rules['PS1']}{ps1_label}")
print()
print("BP1 (missense in a gene where only LOF is a known mechanism):")
if bp1_rules['reason']:
    print(f"  Reason: {bp1_rules['reason']}")
print(f"  BP1 (supporting benign): {bp1_rules['BP1']}")


PS1 (same amino acid change, previously P/LP in ClinVar):
  Position:       None
  Current change: None
  Reason:         not a missense variant
  PS1: False

BP1 (missense in a gene where only LOF is a known mechanism):
  Reason: BP1 marked Not Applicable in CSpec for this gene
  BP1 (supporting benign): False


In [12]:
pm4_rules = apply_pm4_bp3_rule(context)
print(f"Repeat class: {pm4_rules['repeat_class'] or '(none)'}")
if pm4_rules['reason']:
    print(f"Reason:       {pm4_rules['reason']}")
print()
print(f"PM4 (moderate pathogenic, protein length change): {pm4_rules['PM4']}")
print(f"BP3 (supporting benign, in-frame indel in repeat):  {pm4_rules['BP3']}")


Repeat class: (none)
Reason:       not an in-frame indel or stop-loss variant

PM4 (moderate pathogenic, protein length change): False
BP3 (supporting benign, in-frame indel in repeat):  False


In [13]:
bp7_rules = apply_bp7_rule(context)
print(f"SpliceAI score: {bp7_rules['spliceai_score']}")
print(f"BP7 threshold:  < {bp7_rules['bp7_threshold']}")
if bp7_rules['reason']:
    print(f"Reason:         {bp7_rules['reason']}")
print()
print(f"BP7 (supporting benign, no predicted splicing impact): {bp7_rules['BP7']}")


SpliceAI score: None
BP7 threshold:  < 0.1
Reason:         not a synonymous variant (intronic branch not implemented)

BP7 (supporting benign, no predicted splicing impact): False


In [14]:
ps2pm6_rules = apply_ps2_pm6_rule(context)
print(f"De novo status: {ps2pm6_rules['de_novo_status']}")
if ps2pm6_rules['reason']:
    print(f"Reason:         {ps2pm6_rules['reason']}")
print()
print(f"PS2 (strong pathogenic, confirmed de novo):   {ps2pm6_rules['PS2']}")
print(f"PM6 (moderate pathogenic, assumed de novo):   {ps2pm6_rules['PM6']}")


De novo status: confirmed

PS2 (strong pathogenic, confirmed de novo):   True
PM6 (moderate pathogenic, assumed de novo):   False


In [ ]:
ps4_rules = apply_ps4_rule(context)
print(f"Disease cohort checked: {(context['manual_evidence'].get('disease_context') or 'none').upper()}")
if ps4_rules['reason']:
    print(f"Reason:         {ps4_rules['reason']}")
if ps4_rules['case_ac'] is not None:
    print(f"Case AC/AN:     {ps4_rules['case_ac']}/{ps4_rules['case_an']}")
if ps4_rules['control_ac'] is not None:
    print(f"Control AC/AN:  {ps4_rules['control_ac']}/{ps4_rules['control_an']}  (gnomAD joint)")
if ps4_rules['odds_ratio'] is not None:
    print(f"Odds ratio:     {ps4_rules['odds_ratio']:.2f}  (95% CI {ps4_rules['ci_lower']:.2f}-{ps4_rules['ci_upper']:.2f})")
    thresholds_str = "   ".join(f"{label.capitalize()} >= {t}" for t, label in _PS4_CI_LOWER_THRESHOLDS)
    print(f"Strength thresholds (lower 95% CI):  {thresholds_str}")
print()
ps4_label = f"  [{ps4_rules['PS4_strength']}]" if ps4_rules['PS4'] else ""
print(f"PS4 (case-control enrichment): {ps4_rules['PS4']}{ps4_label}")


In [ ]:
ps3_rules = apply_ps3_bs3_rule(context)
print(f"Matched cDNA change: {context['gene_symbol']} {(context.get('chosen_transcript') or {}).get('hgvsc','')}")
if ps3_rules['reason']:
    print(f"Reason:         {ps3_rules['reason']}")
if ps3_rules['matched_disease']:
    print(f"Curation disease context: {ps3_rules['matched_disease']}")
if ps3_rules['evidence']:
    for ev, src in zip(ps3_rules['evidence'], ps3_rules['sources']):
        print(f"Evidence:       {ev}  ({src})")
print()
ps3_label = f"  [{ps3_rules['PS3_strength']}]" if ps3_rules['PS3'] else ""
bs3_label = f"  [{ps3_rules['BS3_strength']}]" if ps3_rules['BS3'] else ""
print(f"PS3 (functional studies, damaging):    {ps3_rules['PS3']}{ps3_label}")
print(f"BS3 (functional studies, no damage):   {ps3_rules['BS3']}{bs3_label}")


In [ ]:
curator_rules = apply_curator_only_rules(context)
print("Curator-only evidence (PM3, PP1, PP4, BS2, BS4, BP2, BP5 -- sourced from")
print("the acmg_curations literature table; not computable from sequence/frequency data,")
print("same as in the original Perl classifier).")
if curator_rules['reason']:
    print(f"Reason: {curator_rules['reason']}")
print()
for _code in _CURATOR_ONLY_CODES:
    fired = curator_rules[_code]
    label = f"  [{curator_rules[f'{_code}_strength']}]" if fired else ""
    print(f"{_code:<5} {fired}{label}")
    if fired:
        for ev, src in zip(curator_rules['evidence'][_code], curator_rules['sources'][_code]):
            print(f"      {ev}  ({src})")


In [ ]:
result = classify_variant(context)
freq   = result["freq_detail"]
comp   = result["comp_detail"]
pvs1   = result["pvs1_detail"]
pm1    = result["pm1_detail"]
ps1    = result["ps1_detail"]
pm5    = result["pm5_detail"]
bp1    = result["bp1_detail"]
pm4    = result["pm4_detail"]
bp7    = result["bp7_detail"]
ps2pm6 = result["ps2pm6_detail"]
ps4    = result["ps4_detail"]
ps3    = result["ps3_detail"]
curator = result["curator_detail"]
clin   = result["clin_detail"]

print("=" * 54)
print(f"  VARIANT:    {context['annotation'].get('input','')}")
print(f"  GENE:       {context['gene_symbol']}")
print(f"  DIAGNOSIS:  {disease_long_name}" + (f" ({disease_short_code})" if disease_short_code else ""))
print("=" * 54)
print()
print("Evidence fired:")
if result["evidence_codes"]:
    for code in result["evidence_codes"]:
        print(f"  {code}")
else:
    print("  (none)")
print()
print(f"Tavtigian combined score:   {result['combined_score']}")
print(f"Posterior probability:      {result['posterior_probability']:.4f}")
print()
_subtier_suffix = f"  ({result['vus_subtier']}, {result['combined_score']} pts)" if result.get('vus_subtier') else ""
print(f">>> CLASSIFICATION: {result['classification']}{_subtier_suffix} <<<")
print("=" * 54)
print()
print("Evidence detail:")
_pvs1_ttn = f"  ttn_gate={pvs1['ttn_gate']['gate_pass']} ({pvs1['ttn_gate']['reason'] or 'ok'})" if pvs1.get('ttn_gate') else ""
print(f"  PVS1  null={pvs1['is_null_variant']}  LOF_in_G2P={pvs1['lof_mechanism']}  => {pvs1['PVS1']}{_pvs1_ttn}")
print(f"  BA1   af={freq['af_used']}  thresh>={freq['ba1_threshold']}  => {freq['BA1']}")
print(f"  BS1   af={freq['af_used']}  thresh>={freq['bs1_threshold']}  => {freq['BS1']}")
print(f"  PM2   af={freq['af_used']}  thresh<={freq['pm2_max']}  strength={freq['pm2_strength']}  => {freq['PM2']}")
print(f"  PS2   de_novo_status={ps2pm6['de_novo_status']}  => {ps2pm6['PS2']}")
print(f"  PS1   change={ps1['current_change']}  strength={ps1['PS1_strength']}  hits={len(ps1['ps1_hits'])}  => {ps1['PS1']}")
print(f"  PM5   change={pm5['current_change']}  strength={pm5['PM5_strength']}  hits={len(pm5['pm5_hits'])}  => {pm5['PM5']}")
ps4_ci = f"CI=[{ps4['ci_lower']:.2f}-{ps4['ci_upper']:.2f}]" if ps4['ci_lower'] is not None else "CI=None"
ps4_thresh = "/".join(f"{label[:4]}>={t}" for t, label in _PS4_CI_LOWER_THRESHOLDS)
print(f"  PS4   disease={context['manual_evidence'].get('disease_context')}  OR={ps4['odds_ratio']}  {ps4_ci}  thresh(lower CI)={ps4_thresh}  strength={ps4['PS4_strength']}  => {ps4['PS4']}")
print(f"  PS3   source={ps3['sources']}  strength={ps3['PS3_strength']}  => {ps3['PS3']}")
print(f"  BS3   source={ps3['sources']}  strength={ps3['BS3_strength']}  => {ps3['BS3']}")
print(f"  PM1   region={pm1['matched_region']}  strength={pm1['PM1_strength']}  => {pm1['PM1']}")
print(f"  PP2   region={pm1['matched_region']}  strength={pm1['PP2_strength']}  => {pm1['PP2']}")
print(f"  PM4   repeat_class={pm4['repeat_class']}  => {pm4['PM4']}")
print(f"  BP3   repeat_class={pm4['repeat_class']}  => {pm4['BP3']}")
print(f"  PM6   de_novo_status={ps2pm6['de_novo_status']}  => {ps2pm6['PM6']}")
print(f"  BP1   => {bp1['BP1']}  ({bp1['reason'] or 'mechanism is LOF-only'})")
print(f"  PP3   REVEL={comp['revel_score']}  thresh>={comp['pp3_threshold']}  => {comp['PP3']}")
print(f"  BP4   REVEL={comp['revel_score']}  thresh<={comp['bp4_threshold']}  => {comp['BP4']}")
print(f"  BP7   SpliceAI={bp7['spliceai_score']}  thresh<{bp7['bp7_threshold']}  => {bp7['BP7']}")
for _code in _CURATOR_ONLY_CODES:
    print(f"  {_code:<5} curated={curator[_code]}  strength={curator[f'{_code}_strength']}  => {curator[_code]}")
print(f"  [Source: {freq['cspec_source']}]")
print()
print(f"  (for reference, not scored) ClinVar significance={clin['clinvar_significance']}"
      f"  PP5={clin['PP5']}  BP6={clin['BP6']}  -- excluded per 2018 SVI recommendation")

# --------------------------------------------------
# Per-variant classification JSON (distinct from the annotation JSON saved
# earlier) -- resembles what a "classify this variant" API response would
# look like, rather than the flat semicolon-joined strings the original
# Perl tool returned. Written for every run (interactive or via
# validation.ipynb's %run -i main1.ipynb), one file per variant, same
# outputs/ folder and basename as the annotation JSON/TSV.
def _json_safe(obj):
    """Recursively convert numpy/pandas scalar types to plain Python types."""
    if isinstance(obj, dict):
        return {k: _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_safe(v) for v in obj]
    if hasattr(obj, "item") and not isinstance(obj, (str, bytes)):
        try:
            return obj.item()
        except (ValueError, AttributeError):
            pass
    return obj

gdp = context["gene_disease_pair"]

classification_output = {
    "variant": context["annotation"].get("input", ""),
    "gene": context["gene_symbol"],
    "diagnosis": {
        "short_code": disease_short_code,
        "long_name": disease_long_name,
        "gene_disease_pair_confirmed": gdp["pair_found"],
        "reason": gdp.get("reason"),
    },
    "classification": {
        "label": result["classification"],
        "vus_subtier": result.get("vus_subtier"),
        "posterior_probability": result["posterior_probability"],
        "tavtigian_combined_score": result["combined_score"],
        "evidence_codes_fired": [c[0] if isinstance(c, tuple) else c for c in result["evidence_codes"]],
    },
    "evidence_detail": {
        "PVS1": {"fired": pvs1["PVS1"], "is_null_variant": pvs1["is_null_variant"], "lof_mechanism": pvs1["lof_mechanism"], "ttn_gate": pvs1.get("ttn_gate")},
        "BA1":  {"fired": freq["BA1"], "af_used": freq["af_used"], "threshold": freq["ba1_threshold"]},
        "BS1":  {"fired": freq["BS1"], "af_used": freq["af_used"], "threshold": freq["bs1_threshold"]},
        "PM2":  {"fired": freq["PM2"], "af_used": freq["af_used"], "threshold": freq["pm2_max"]},
        "PS1":  {"fired": ps1["PS1"], "strength": ps1["PS1_strength"], "current_change": ps1["current_change"], "hits": ps1["ps1_hits"]},
        "PM5":  {"fired": pm5["PM5"], "strength": pm5["PM5_strength"], "current_change": pm5["current_change"], "hits": pm5["pm5_hits"]},
        "PS2":  {"fired": ps2pm6["PS2"], "de_novo_status": ps2pm6["de_novo_status"]},
        "PM6":  {"fired": ps2pm6["PM6"], "de_novo_status": ps2pm6["de_novo_status"]},
        "PS3":  {"fired": ps3["PS3"], "strength": ps3["PS3_strength"], "sources": ps3["sources"]},
        "BS3":  {"fired": ps3["BS3"], "strength": ps3["BS3_strength"], "sources": ps3["sources"]},
        "PS4":  {"fired": ps4["PS4"], "strength": ps4["PS4_strength"], "odds_ratio": ps4["odds_ratio"],
                 "ci_lower": ps4["ci_lower"], "ci_upper": ps4["ci_upper"],
                 "case_ac": ps4["case_ac"], "case_an": ps4["case_an"],
                 "control_ac": ps4["control_ac"], "control_an": ps4["control_an"]},
        "PM1":  {"fired": pm1["PM1"], "strength": pm1["PM1_strength"], "matched_region": pm1["matched_region"]},
        "PP2":  {"fired": pm1["PP2"], "strength": pm1["PP2_strength"], "matched_region": pm1["matched_region"]},
        "PM4":  {"fired": pm4["PM4"], "repeat_class": pm4["repeat_class"]},
        "BP3":  {"fired": pm4["BP3"], "repeat_class": pm4["repeat_class"]},
        "BP1":  {"fired": bp1["BP1"], "reason": bp1["reason"]},
        "PP3":  {"fired": comp["PP3"], "revel_score": comp["revel_score"], "threshold": comp["pp3_threshold"]},
        "BP4":  {"fired": comp["BP4"], "revel_score": comp["revel_score"], "threshold": comp["bp4_threshold"]},
        "BP7":  {"fired": bp7["BP7"], "spliceai_score": bp7["spliceai_score"], "threshold": bp7["bp7_threshold"]},
        **{
            _code: {"fired": curator[_code], "strength": curator[f"{_code}_strength"],
                     "evidence": curator["evidence"].get(_code, []), "sources": curator["sources"].get(_code, [])}
            for _code in _CURATOR_ONLY_CODES
        },
    },
    "manual_evidence": context["manual_evidence"],
    "clinvar_reference": {
        "significance": clin["clinvar_significance"],
        "PP5": clin["PP5"],
        "BP6": clin["BP6"],
        "note": "excluded from scoring per 2018 SVI recommendation",
    },
}

classification_json_path = Path(f"outputs/{basename}_classification.json")
with classification_json_path.open("w", encoding="utf-8") as f:
    json.dump(_json_safe(classification_output), f, indent=2)
print()
print("Saved classification JSON:", classification_json_path)
